# UFC Winner Modeling and Betting Backtest

This notebook builds time-aware fight prediction models from `df.csv`, scores them with probability-based confidence, and evaluates whether they can beat the betting market on a rolling out-of-sample backtest.

Key rules used here:
- No post-fight leakage columns in the feature set.
- No fighter-name memorization features.
- Time-ordered validation and test splits only.
- Weight-class-specific models are allowed, but no extra datasets are written to disk.
- Confidence is the predicted probability assigned to the model's chosen winner.


In [1]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import (
    AdaBoostClassifier,
    ExtraTreesClassifier,
    GradientBoostingClassifier,
    HistGradientBoostingClassifier,
    RandomForestClassifier,
)
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, brier_score_loss, log_loss
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from xgboost import XGBClassifier

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 160)
pd.set_option("display.float_format", lambda x: f"{x:0.4f}")

ROOT = Path.cwd()
if not (ROOT / "data" / "df.csv").exists():
    ROOT = ROOT.parent

DATA_PATH = ROOT / "data" / "df.csv"
raw_df = pd.read_csv(DATA_PATH, parse_dates=["Date"]).sort_values("Date").reset_index(drop=True)

profile = pd.DataFrame(
    {
        "metric": [
            "rows",
            "columns",
            "date_min",
            "date_max",
            "rows_2018_plus",
            "male_rows",
            "female_rows",
        ],
        "value": [
            len(raw_df),
            raw_df.shape[1],
            raw_df["Date"].min().date(),
            raw_df["Date"].max().date(),
            int((raw_df["Date"] >= pd.Timestamp("2018-01-01")).sum()),
            int((raw_df["Gender"] == "MALE").sum()),
            int((raw_df["Gender"] == "FEMALE").sum()),
        ],
    }
).set_index("metric")

display(profile)
display(raw_df["WeightClass"].value_counts().rename("fights"))


C:\Users\Anthony\anaconda3\lib\site-packages\xgboost\compat.py:36: FutureWarning: pandas.Int64Index is deprecated and will be removed from pandas in a future version. Use pandas.Index with the appropriate dtype instead.
  from pandas import MultiIndex, Int64Index


,value
metric,
rows,6383
columns,52
date_min,2010-03-21
date_max,2024-12-14
rows_2018_plus,3414
male_rows,5597
female_rows,786


Lightweight              1063
Welterweight              993
Middleweight              758
Featherweight             737
Bantamweight              675
Light Heavyweight         492
Heavyweight               473
Flyweight                 347
Women's Strawweight       308
Women's Flyweight         237
Women's Bantamweight      212
Catch Weight               59
Women's Featherweight      29
Name: fights, dtype: int64

## Modeling Setup

The target is `RedWinner`, treated as a red-corner win probability problem. The following columns are excluded because they are not available before the fight or would create unrealistic leakage:

- `RedWinner`
- `RedReturn`
- `BlueReturn`
- `Finish`
- `FinishRound`
- `RedFighter`
- `BlueFighter`

The betting backtest uses expected value against the posted odds. For each fight, the model can either:
- bet red,
- bet blue,
- or pass if neither side clears the tuned EV threshold.

Return rate is the mean amount returned per 1 unit staked, so values above `1.0` are profitable.


In [2]:
LEAKAGE_COLS = ["RedWinner", "RedReturn", "BlueReturn", "Finish", "FinishRound"]
IDENTITY_COLS = ["RedFighter", "BlueFighter"]
BASE_FEATURE_COLS = [c for c in raw_df.columns if c not in LEAKAGE_COLS + IDENTITY_COLS]
RETURN_COLS = ["RedReturn", "BlueReturn", "RedOdds", "BlueOdds"]
CATEGORICAL_COLS = ["WeightClass", "Gender", "BlueStance", "RedStance"]
GLOBAL_START_DATES = ["2010-01-01", "2014-01-01", "2016-01-01", "2018-01-01"]
TEST_YEARS = [2021, 2022, 2023, 2024]
THRESHOLD_GRID = np.round(np.linspace(-0.02, 0.20, 45), 3)

DEFAULT_MODEL_PARAMS = {
    "logreg": {"C": 0.5},
    "rf": {"n_estimators": 400, "min_samples_leaf": 3, "max_depth": None},
    "extra_trees": {"n_estimators": 400, "min_samples_leaf": 3, "max_depth": None},
    "gb": {"n_estimators": 250, "learning_rate": 0.04, "max_depth": 2},
    "hist_gb": {"max_depth": 5, "learning_rate": 0.03, "max_iter": 350},
    "ada": {"n_estimators": 400, "learning_rate": 0.03},
    "xgb": {"n_estimators": 300, "max_depth": 3, "learning_rate": 0.03},
}

ESTIMATOR_PARAM_GRIDS = {
    "logreg": [
        {"C": 0.05},
        {"C": 0.10},
        {"C": 0.50},
        {"C": 1.50},
    ],
    "rf": [
        {"n_estimators": 300, "min_samples_leaf": 2, "max_depth": None},
        {"n_estimators": 500, "min_samples_leaf": 3, "max_depth": None},
    ],
    "extra_trees": [
        {"n_estimators": 300, "min_samples_leaf": 2, "max_depth": None},
        {"n_estimators": 500, "min_samples_leaf": 3, "max_depth": None},
    ],
    "gb": [
        {"n_estimators": 200, "learning_rate": 0.05, "max_depth": 2},
        {"n_estimators": 300, "learning_rate": 0.03, "max_depth": 2},
        {"n_estimators": 250, "learning_rate": 0.04, "max_depth": 3},
    ],
    "hist_gb": [
        {"max_depth": 3, "learning_rate": 0.05, "max_iter": 250},
        {"max_depth": 5, "learning_rate": 0.03, "max_iter": 350},
        {"max_depth": 6, "learning_rate": 0.02, "max_iter": 500},
    ],
    "ada": [
        {"n_estimators": 200, "learning_rate": 0.05},
        {"n_estimators": 400, "learning_rate": 0.03},
        {"n_estimators": 600, "learning_rate": 0.02},
    ],
    "xgb": [
        {"n_estimators": 250, "max_depth": 2, "learning_rate": 0.05},
        {"n_estimators": 350, "max_depth": 3, "learning_rate": 0.03},
    ],
}


def american_to_decimal(odds):
    """Convert American odds to decimal odds so expected-value math is easier. Returns a NumPy array of decimal prices."""
    odds = np.asarray(odds, dtype=float)
    return np.where(odds > 0, 1 + odds / 100.0, 1 + 100.0 / np.abs(odds))


def prepare_feature_frame(frame):
    """Clean the raw fight rows and add bookmaker-derived probability features that are known before the fight. Returns a model-ready DataFrame with the original pre-fight columns plus engineered odds features."""
    working = frame.copy()
    if "Date" not in working.columns:
        working["Date"] = pd.NaT

    X = working.reindex(columns=BASE_FEATURE_COLS).copy()

    for col in ["BlueStance", "RedStance"]:
        if col in X.columns:
            X[col] = X[col].replace({"Switch ": "Switch"})

    red_decimal = american_to_decimal(X["RedOdds"])
    blue_decimal = american_to_decimal(X["BlueOdds"])
    red_implied = 1 / red_decimal
    blue_implied = 1 / blue_decimal

    X["RedDecimalOdds"] = red_decimal
    X["BlueDecimalOdds"] = blue_decimal
    X["RedImpliedProb"] = red_implied
    X["BlueImpliedProb"] = blue_implied
    X["RedNoVigProb"] = red_implied / (red_implied + blue_implied)
    X["BlueNoVigProb"] = blue_implied / (red_implied + blue_implied)
    X["BookHold"] = red_implied + blue_implied - 1

    return X


def make_one_hot_encoder():
    """Build a dense one-hot encoder that works on both newer and older scikit-learn versions. Returns a configured `OneHotEncoder` instance."""
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=False)


def build_preprocessor(feature_cols):
    """Split numeric and categorical columns, then impute and scale them consistently inside a single preprocessing object. Returns a `ColumnTransformer` that can be reused across estimators."""
    numeric_cols = [c for c in feature_cols if c not in CATEGORICAL_COLS]
    return ColumnTransformer(
        transformers=[
            (
                "num",
                Pipeline(
                    steps=[
                        ("imputer", SimpleImputer(strategy="median")),
                        ("scaler", StandardScaler()),
                    ]
                ),
                numeric_cols,
            ),
            (
                "cat",
                Pipeline(
                    steps=[
                        ("imputer", SimpleImputer(strategy="most_frequent")),
                        ("onehot", make_one_hot_encoder()),
                    ]
                ),
                [c for c in CATEGORICAL_COLS if c in feature_cols],
            ),
        ],
        remainder="drop",
    )


def build_estimator(model_name, model_params=None):
    """Create the requested estimator family and merge in any tuned parameters for that run. Returns an unfitted scikit-learn or XGBoost classifier."""
    merged_params = {**DEFAULT_MODEL_PARAMS[model_name], **(model_params or {})}

    if model_name == "logreg":
        return LogisticRegression(max_iter=3000, random_state=42, **merged_params)
    if model_name == "rf":
        return RandomForestClassifier(random_state=42, n_jobs=-1, **merged_params)
    if model_name == "extra_trees":
        return ExtraTreesClassifier(random_state=42, n_jobs=-1, **merged_params)
    if model_name == "gb":
        return GradientBoostingClassifier(random_state=42, **merged_params)
    if model_name == "hist_gb":
        return HistGradientBoostingClassifier(random_state=42, **merged_params)
    if model_name == "ada":
        return AdaBoostClassifier(random_state=42, **merged_params)
    if model_name == "xgb":
        return XGBClassifier(
            random_state=42,
            eval_metric="logloss",
            n_jobs=4,
            subsample=0.8,
            colsample_bytree=0.8,
            reg_lambda=1.0,
            min_child_weight=2,
            **merged_params,
        )
    raise ValueError(f"Unknown model name: {model_name}")


def build_pipeline(feature_cols, model_name, model_params=None):
    """Combine preprocessing and estimator construction into one reusable training pipeline. Returns an unfitted `Pipeline` that can be trained on a fight table."""
    return Pipeline(
        steps=[
            ("prep", build_preprocessor(feature_cols)),
            ("model", build_estimator(model_name, model_params)),
        ]
    )


def format_params(model_params):
    """Turn a parameter dictionary into a compact display label for notebook tables. Returns a sorted string such as `C=0.5, max_depth=3`."""
    if not model_params:
        return "default"
    return ", ".join(f"{key}={value}" for key, value in sorted(model_params.items()))


def evaluate_betting_strategy(proba_red, odds_frame, threshold):
    """Score one betting rule by turning red-win probabilities into side selections and pass/bet decisions. Returns a dictionary with bet count, realized returns, and average return metrics."""
    red_decimal = american_to_decimal(odds_frame["RedOdds"])
    blue_decimal = american_to_decimal(odds_frame["BlueOdds"])

    ev_red = proba_red * red_decimal - 1
    ev_blue = (1 - proba_red) * blue_decimal - 1
    choose_red = ev_red >= ev_blue
    max_ev = np.where(choose_red, ev_red, ev_blue)
    bet_mask = max_ev > threshold

    realized_return = np.where(
        choose_red,
        odds_frame["RedReturn"].to_numpy(),
        odds_frame["BlueReturn"].to_numpy(),
    )
    placed_returns = realized_return[bet_mask]

    return {
        "bets": int(placed_returns.size),
        "returns": placed_returns,
        "return_rate": float(np.mean(placed_returns)) if placed_returns.size else np.nan,
        "profit_per_bet": float(np.mean(placed_returns - 1)) if placed_returns.size else np.nan,
        "threshold": float(threshold),
    }


def tune_threshold(proba_red, odds_frame, min_bets):
    """Search the EV threshold grid for the most profitable validation rule without dropping below a minimum number of bets. Returns the chosen threshold and its full validation summary."""
    best = None
    for threshold in THRESHOLD_GRID:
        stats = evaluate_betting_strategy(proba_red, odds_frame, threshold)
        if stats["bets"] < min_bets:
            continue
        score = (stats["return_rate"], stats["bets"])
        if best is None or score > (best["return_rate"], best["bets"]):
            best = stats
    if best is None:
        best = evaluate_betting_strategy(proba_red, odds_frame, 0.0)
    return best["threshold"], best


def choose_weight_class_backtest_config(frame, weight_class):
    """Scale the minimum train, validation, and test sizes to the division's sample size so every weight class can still be modeled. Returns a dictionary of split thresholds for the rolling backtest."""
    total_fights = int((frame["WeightClass"] == weight_class).sum())

    if total_fights >= 300:
        return {"min_train": 120, "min_val": 20, "min_test": 20, "min_val_bets": 8}
    if total_fights >= 150:
        return {"min_train": 80, "min_val": 12, "min_test": 12, "min_val_bets": 5}
    if total_fights >= 60:
        return {"min_train": 30, "min_val": 6, "min_test": 6, "min_val_bets": 3}
    return {"min_train": 15, "min_val": 3, "min_test": 3, "min_val_bets": 1}


def rolling_backtest(
    frame,
    model_name,
    model_params=None,
    start_date=None,
    weight_class=None,
    min_train=500,
    min_val=100,
    min_test=100,
    min_val_bets=30,
):
    """Run a chronological backtest for one candidate model.

    The function optionally filters to a date range or a single weight class, rebuilds the pre-fight feature matrix, then walks year by year through train/validate/test splits. Each loop fits the model on the training window, tunes the EV betting threshold on the validation year, scores the next year out of sample, and finally returns aggregated prediction and ROI metrics across every usable split.
    """
    data = frame.copy()
    if start_date is not None:
        data = data[data["Date"] >= pd.Timestamp(start_date)]
    if weight_class is not None:
        data = data[data["WeightClass"] == weight_class]
    data = data.sort_values("Date").reset_index(drop=True)
    if data.empty:
        return None

    X = prepare_feature_frame(data)
    y = data["RedWinner"].astype(int)
    feature_cols = [c for c in X.columns if c != "Date"]

    all_returns = []
    all_truth = []
    all_proba = []
    total_bets = 0
    yearly_details = []
    completed_splits = 0

    for test_year in TEST_YEARS:
        # 1. Define the chronological training window, validation window, and truly out-of-sample test year.
        val_start = pd.Timestamp(f"{test_year - 1}-01-01")
        test_start = pd.Timestamp(f"{test_year}-01-01")
        test_end = pd.Timestamp(f"{test_year + 1}-01-01")

        train_mask = data["Date"] < val_start
        val_mask = (data["Date"] >= val_start) & (data["Date"] < test_start)
        test_mask = (data["Date"] >= test_start) & (data["Date"] < test_end)

        # 2. Skip years that are too thin to support a meaningful train/validation/test split.
        if train_mask.sum() < min_train or val_mask.sum() < min_val or test_mask.sum() < min_test:
            continue

        # 3. Fit the candidate model on past fights only, then create probability forecasts for validation and test fights.
        pipeline = build_pipeline(feature_cols, model_name, model_params=model_params)
        pipeline.fit(X.loc[train_mask, feature_cols], y.loc[train_mask])

        val_proba = pipeline.predict_proba(X.loc[val_mask, feature_cols])[:, 1]
        test_proba = pipeline.predict_proba(X.loc[test_mask, feature_cols])[:, 1]

        # 4. Tune the EV threshold on the validation year, then lock it in and apply it to the next year.
        threshold, _ = tune_threshold(val_proba, data.loc[val_mask, RETURN_COLS], min_bets=min_val_bets)
        test_stats = evaluate_betting_strategy(test_proba, data.loc[test_mask, RETURN_COLS], threshold)

        # 5. Store the realized returns and the raw class probabilities so we can summarize both ROI and calibration later.
        if test_stats["bets"]:
            all_returns.append(test_stats["returns"])
            total_bets += test_stats["bets"]
        all_truth.append(y.loc[test_mask].to_numpy())
        all_proba.append(test_proba)
        completed_splits += 1

        yearly_details.append(
            {
                "test_year": test_year,
                "threshold": threshold,
                "bets": test_stats["bets"],
                "return_rate": test_stats["return_rate"],
            }
        )

    if not all_returns:
        return None

    flat_returns = np.concatenate(all_returns)
    flat_truth = np.concatenate(all_truth)
    flat_proba = np.concatenate(all_proba)

    return {
        "model": model_name,
        "model_params": model_params or {},
        "params_label": format_params(model_params or {}),
        "start_date": start_date,
        "weight_class": weight_class,
        "bets": int(total_bets),
        "return_rate": float(np.mean(flat_returns)),
        "profit_per_bet": float(np.mean(flat_returns - 1)),
        "accuracy": float(accuracy_score(flat_truth, flat_proba >= 0.5)),
        "brier": float(brier_score_loss(flat_truth, flat_proba)),
        "logloss": float(log_loss(flat_truth, flat_proba, labels=[0, 1])),
        "splits": completed_splits,
        "yearly_details": yearly_details,
    }


def favorite_baseline(frame, weight_class=None, start_date="2021-01-01", end_date="2025-01-01"):
    """Benchmark a plain favorite/underdog approach over the requested slice of fights. Returns simple average return rates for favorites and underdogs."""
    data = frame.copy()
    if weight_class is not None:
        data = data[data["WeightClass"] == weight_class]
    data = data[(data["Date"] >= pd.Timestamp(start_date)) & (data["Date"] < pd.Timestamp(end_date))]
    favorite_returns = np.where(data["RedOdds"] < data["BlueOdds"], data["RedReturn"], data["BlueReturn"])
    underdog_returns = np.where(data["RedOdds"] >= data["BlueOdds"], data["RedReturn"], data["BlueReturn"])
    return {
        "fights": int(len(data)),
        "favorite_return_rate": float(np.mean(favorite_returns)),
        "underdog_return_rate": float(np.mean(underdog_returns)),
    }


In [3]:
global_results = []
for start_date in GLOBAL_START_DATES:
    for model_name in ESTIMATOR_PARAM_GRIDS:
        result = rolling_backtest(
            raw_df,
            model_name,
            model_params=DEFAULT_MODEL_PARAMS[model_name],
            start_date=start_date,
        )
        if result is not None:
            global_results.append(result)

global_results = pd.DataFrame(global_results).sort_values(["return_rate", "bets"], ascending=[False, False]).reset_index(drop=True)

display(
    global_results[
        ["model", "params_label", "start_date", "bets", "return_rate", "profit_per_bet", "accuracy", "brier", "logloss"]
    ]
)

display(
    global_results.query("return_rate > 1")[
        ["model", "params_label", "start_date", "bets", "return_rate", "profit_per_bet", "accuracy", "brier"]
    ]
)


,model,params_label,start_date,bets,return_rate,profit_per_bet,accuracy,brier,logloss
0,logreg,C=0.5,2014-01-01,182,1.1881,0.1881,0.6723,0.2086,0.6035
1,logreg,C=0.5,2016-01-01,290,1.1811,0.1811,0.6767,0.2086,0.6034
2,logreg,C=0.5,2010-01-01,242,1.0868,0.0868,0.6753,0.2089,0.6040
3,logreg,C=0.5,2018-01-01,703,1.0483,0.0483,0.6673,0.2102,0.6065
4,xgb,"learning_rate=0.03, max_depth=3, n_estimators=300",2016-01-01,1298,1.0092,0.0092,0.6723,0.2105,0.6087
5,gb,"learning_rate=0.04, max_depth=2, n_estimators=250",2014-01-01,504,0.9965,-0.0035,0.6658,0.2107,0.6088
6,hist_gb,"learning_rate=0.03, max_depth=5, max_iter=350",2014-01-01,958,0.9889,-0.0111,0.6703,0.2153,0.6219
7,gb,"learning_rate=0.04, max_depth=2, n_estimators=250",2010-01-01,455,0.9785,-0.0215,0.6728,0.2106,0.6082
8,hist_gb,"learning_rate=0.03, max_depth=5, max_iter=350",2016-01-01,1433,0.9720,-0.0280,0.6525,0.2191,0.6318
9,xgb,"learning_rate=0.03, max_depth=3, n_estimators=300",2018-01-01,639,0.9661,-0.0339,0.6683,0.2124,0.6134


,model,params_label,start_date,bets,return_rate,profit_per_bet,accuracy,brier
0,logreg,C=0.5,2014-01-01,182,1.1881,0.1881,0.6723,0.2086
1,logreg,C=0.5,2016-01-01,290,1.1811,0.1811,0.6767,0.2086
2,logreg,C=0.5,2010-01-01,242,1.0868,0.0868,0.6753,0.2089
3,logreg,C=0.5,2018-01-01,703,1.0483,0.0483,0.6673,0.2102
4,xgb,"learning_rate=0.03, max_depth=3, n_estimators=300",2016-01-01,1298,1.0092,0.0092,0.6723,0.2105


In [4]:
all_weight_classes = raw_df["WeightClass"].value_counts().index.tolist()

weight_results = []
for weight_class in all_weight_classes:
    class_config = choose_weight_class_backtest_config(raw_df, weight_class)
    fight_count = int((raw_df["WeightClass"] == weight_class).sum())

    for model_name, param_grid in ESTIMATOR_PARAM_GRIDS.items():
        for model_params in param_grid:
            result = rolling_backtest(
                raw_df,
                model_name,
                model_params=model_params,
                weight_class=weight_class,
                **class_config,
            )
            if result is not None:
                result["fight_count"] = fight_count
                result["config_label"] = format_params(class_config)
                weight_results.append(result)

weight_results = pd.DataFrame(weight_results)
missing_weight_classes = sorted(set(all_weight_classes) - set(weight_results["weight_class"].unique()))
if missing_weight_classes:
    raise ValueError(f"Missing weight-class results for: {missing_weight_classes}")

weight_results = weight_results.sort_values(
    ["weight_class", "return_rate", "bets", "accuracy", "brier"],
    ascending=[True, False, False, False, True],
).reset_index(drop=True)

# Ten bets is a lighter floor than before, but it still removes the noisiest ultra-small samples from the profitable table.
profitability_bet_floor = 10
profitable_weight_results = weight_results.query("return_rate > 1 and bets >= @profitability_bet_floor").copy()

display(
    weight_results[
        [
            "weight_class",
            "fight_count",
            "model",
            "params_label",
            "splits",
            "bets",
            "return_rate",
            "profit_per_bet",
            "accuracy",
            "brier",
            "logloss",
        ]
    ]
)

display(
    profitable_weight_results[
        [
            "weight_class",
            "fight_count",
            "model",
            "params_label",
            "splits",
            "bets",
            "return_rate",
            "profit_per_bet",
            "accuracy",
            "brier",
        ]
    ]
)


,weight_class,fight_count,model,params_label,splits,bets,return_rate,profit_per_bet,accuracy,brier,logloss
0,Bantamweight,675,gb,"learning_rate=0.04, max_depth=3, n_estimators=250",4,151,0.9559,-0.0441,0.6438,0.2242,0.6478
1,Bantamweight,675,extra_trees,"max_depth=None, min_samples_leaf=2, n_estimato...",4,149,0.9503,-0.0497,0.6824,0.2025,0.5901
2,Bantamweight,675,hist_gb,"learning_rate=0.02, max_depth=6, max_iter=500",4,164,0.9364,-0.0636,0.6567,0.2355,0.7107
3,Bantamweight,675,hist_gb,"learning_rate=0.03, max_depth=5, max_iter=350",4,201,0.9170,-0.0830,0.6609,0.2306,0.6851
4,Bantamweight,675,xgb,"learning_rate=0.05, max_depth=2, n_estimators=250",4,186,0.9061,-0.0939,0.6910,0.2084,0.6098
...,...,...,...,...,...,...,...,...,...,...,...
242,Women's Strawweight,308,rf,"max_depth=None, min_samples_leaf=3, n_estimato...",4,99,0.9081,-0.0919,0.5942,0.2420,0.6902
243,Women's Strawweight,308,gb,"learning_rate=0.03, max_depth=2, n_estimators=300",4,105,0.8978,-0.1022,0.5725,0.2681,0.7545
244,Women's Strawweight,308,rf,"max_depth=None, min_samples_leaf=2, n_estimato...",4,80,0.8446,-0.1554,0.5942,0.2458,0.7007
245,Women's Strawweight,308,extra_trees,"max_depth=None, min_samples_leaf=2, n_estimato...",4,73,0.7258,-0.2742,0.5725,0.2514,0.7047


,weight_class,fight_count,model,params_label,splits,bets,return_rate,profit_per_bet,accuracy,brier
38,Featherweight,737,logreg,C=0.1,4,87,1.3024,0.3024,0.6104,0.2203
39,Featherweight,737,extra_trees,"max_depth=None, min_samples_leaf=3, n_estimato...",4,71,1.2649,0.2649,0.6277,0.2175
40,Featherweight,737,logreg,C=0.05,4,85,1.2482,0.2482,0.6234,0.2196
41,Featherweight,737,logreg,C=0.5,4,115,1.1989,0.1989,0.5931,0.2225
42,Featherweight,737,extra_trees,"max_depth=None, min_samples_leaf=2, n_estimato...",4,96,1.1944,0.1944,0.6407,0.2194
43,Featherweight,737,gb,"learning_rate=0.03, max_depth=2, n_estimators=300",4,117,1.1842,0.1842,0.6407,0.2212
44,Featherweight,737,logreg,C=1.5,4,115,1.1825,0.1825,0.5887,0.2235
45,Featherweight,737,gb,"learning_rate=0.05, max_depth=2, n_estimators=200",4,123,1.1792,0.1792,0.6277,0.2212
46,Featherweight,737,rf,"max_depth=None, min_samples_leaf=3, n_estimato...",4,90,1.1550,0.1550,0.6320,0.2227
47,Featherweight,737,xgb,"learning_rate=0.03, max_depth=3, n_estimators=350",4,130,1.1344,0.1344,0.6104,0.2331


In [5]:
# Pick one tuned model per division. Profitable candidates are preferred first, then higher return rate,
# larger bet samples, stronger classification accuracy, and better Brier score.
best_weight_class_models = (
    weight_results.assign(is_profitable=weight_results["return_rate"] > 1)
    .sort_values(
        ["weight_class", "is_profitable", "return_rate", "bets", "accuracy", "brier", "splits"],
        ascending=[True, False, False, False, False, True, False],
    )
    .groupby("weight_class", as_index=False)
    .head(1)
    .sort_values("weight_class")
    .reset_index(drop=True)
)

final_specs = best_weight_class_models[
    ["weight_class", "model", "model_params", "params_label", "fight_count", "splits", "bets", "return_rate", "profit_per_bet", "accuracy", "brier", "logloss"]
].copy()

baseline_rows = []
for weight_class in final_specs["weight_class"]:
    baseline = favorite_baseline(raw_df, weight_class=weight_class)
    baseline_rows.append({"weight_class": weight_class, **baseline})

baseline_df = pd.DataFrame(baseline_rows)
final_summary = final_specs.merge(baseline_df, on="weight_class", how="left")

display(
    final_summary[
        [
            "weight_class",
            "fight_count",
            "model",
            "params_label",
            "splits",
            "bets",
            "return_rate",
            "profit_per_bet",
            "accuracy",
            "brier",
            "favorite_return_rate",
            "underdog_return_rate",
        ]
    ].sort_values(["return_rate", "bets"], ascending=[False, False])
)


,weight_class,fight_count,model,params_label,splits,bets,return_rate,profit_per_bet,accuracy,brier,favorite_return_rate,underdog_return_rate
5,Light Heavyweight,492,rf,"max_depth=None, min_samples_leaf=2, n_estimato...",4,78,1.3098,0.3098,0.6176,0.2227,0.9525,0.9453
2,Featherweight,737,logreg,C=0.1,4,87,1.3024,0.3024,0.6104,0.2203,0.8948,1.0196
10,Women's Featherweight,29,rf,"max_depth=None, min_samples_leaf=2, n_estimato...",2,6,1.2407,0.2407,0.7778,0.1309,1.1442,0.5407
6,Lightweight,1063,extra_trees,"max_depth=None, min_samples_leaf=3, n_estimato...",4,55,1.1877,0.1877,0.6880,0.2101,0.9995,0.9176
4,Heavyweight,473,logreg,C=0.5,4,101,1.1335,0.1335,0.7163,0.1988,1.0715,0.7009
12,Women's Strawweight,308,logreg,C=0.05,4,97,1.1150,0.1150,0.6014,0.2472,0.9464,1.0082
11,Women's Flyweight,237,ada,"learning_rate=0.02, n_estimators=600",3,80,1.1010,0.1010,0.6105,0.2438,0.9594,0.9562
3,Flyweight,347,hist_gb,"learning_rate=0.02, max_depth=6, max_iter=500",4,110,1.0985,0.0985,0.6812,0.2411,1.0219,0.7894
8,Welterweight,993,rf,"max_depth=None, min_samples_leaf=3, n_estimato...",4,112,1.0009,0.0009,0.6420,0.2055,0.9989,0.8136
1,Catch Weight,59,gb,"learning_rate=0.04, max_depth=3, n_estimators=250",4,30,0.9861,-0.0139,0.5806,0.3913,0.9976,0.7502


In [6]:
def choose_deployment_validation_year(data, preferred_year=2024):
    """Pick the latest year that still leaves earlier fights available for training and enough fights for threshold tuning. Returns an integer year or `None` if the division is too thin for a held-out tuning year."""
    available_years = sorted(data["Date"].dt.year.dropna().unique().tolist(), reverse=True)
    ordered_years = []
    if preferred_year in available_years:
        ordered_years.append(preferred_year)
    ordered_years.extend(year for year in available_years if year != preferred_year)

    for year in ordered_years:
        year_start = pd.Timestamp(f"{year}-01-01")
        year_end = pd.Timestamp(f"{year + 1}-01-01")
        train_count = int((data["Date"] < year_start).sum())
        valid_count = int(((data["Date"] >= year_start) & (data["Date"] < year_end)).sum())
        min_valid = max(1, min(8, valid_count))
        if train_count >= max(10, min(50, len(data) // 4)) and valid_count >= min_valid:
            return int(year)
    return None


def fit_deployment_model(frame, model_name, model_params=None, weight_class=None, start_date=None, validation_year=2024):
    data = frame.copy()
    if start_date is not None:
        data = data[data["Date"] >= pd.Timestamp(start_date)]
    if weight_class is not None:
        data = data[data["WeightClass"] == weight_class]
    data = data.sort_values("Date").reset_index(drop=True)

    X = prepare_feature_frame(data)
    y = data["RedWinner"].astype(int)
    feature_cols = [c for c in X.columns if c != "Date"]

    chosen_validation_year = choose_deployment_validation_year(data, preferred_year=validation_year)
    if chosen_validation_year is None:
        threshold = 0.0
    else:
        valid_start = pd.Timestamp(f"{chosen_validation_year}-01-01")
        valid_end = pd.Timestamp(f"{chosen_validation_year + 1}-01-01")
        tune_train_mask = data["Date"] < valid_start
        tune_valid_mask = (data["Date"] >= valid_start) & (data["Date"] < valid_end)

        tuning_pipeline = build_pipeline(feature_cols, model_name, model_params=model_params)
        tuning_pipeline.fit(X.loc[tune_train_mask, feature_cols], y.loc[tune_train_mask])
        valid_proba = tuning_pipeline.predict_proba(X.loc[tune_valid_mask, feature_cols])[:, 1]
        min_bets = max(1, min(10, int(np.ceil(tune_valid_mask.sum() / 3))))
        threshold, _ = tune_threshold(valid_proba, data.loc[tune_valid_mask, RETURN_COLS], min_bets=min_bets)

    final_pipeline = build_pipeline(feature_cols, model_name, model_params=model_params)
    final_pipeline.fit(X.loc[:, feature_cols], y)

    return {
        "model": final_pipeline,
        "model_name": model_name,
        "model_params": model_params or {},
        "params_label": format_params(model_params or {}),
        "weight_class": weight_class,
        "start_date": start_date,
        "feature_columns": feature_cols,
        "threshold": float(threshold),
        "validation_year_used": chosen_validation_year,
    }


deployment_registry = {}
deployment_rows = []

for row in final_specs.to_dict("records"):
    fitted = fit_deployment_model(
        raw_df,
        row["model"],
        model_params=row["model_params"],
        weight_class=row["weight_class"],
    )
    deployment_registry[row["weight_class"]] = fitted
    deployment_rows.append(
        {
            "segment": row["weight_class"],
            "model": row["model"],
            "params_label": row["params_label"],
            "tuning_year": fitted["validation_year_used"],
            "deployment_threshold": fitted["threshold"],
        }
    )

# Global fallback still exists for unknown divisions, catchweight experiments, or a broader lower-tier product.
global_fallback = fit_deployment_model(
    raw_df,
    "logreg",
    model_params=DEFAULT_MODEL_PARAMS["logreg"],
    start_date="2016-01-01",
)
deployment_registry["__global__"] = global_fallback
deployment_rows.append(
    {
        "segment": "__global__",
        "model": "logreg",
        "params_label": format_params(DEFAULT_MODEL_PARAMS["logreg"]),
        "tuning_year": global_fallback["validation_year_used"],
        "deployment_threshold": global_fallback["threshold"],
    }
)

display(pd.DataFrame(deployment_rows))


def predict_fight(fight_row, registry=deployment_registry):
    if isinstance(fight_row, dict):
        fight_df = pd.DataFrame([fight_row])
    elif isinstance(fight_row, pd.Series):
        fight_df = fight_row.to_frame().T
    else:
        fight_df = fight_row.copy()

    if len(fight_df) != 1:
        raise ValueError("predict_fight expects exactly one fight row.")

    weight_class = fight_df.iloc[0]["WeightClass"]
    bundle = registry.get(weight_class, registry["__global__"])
    feature_frame = prepare_feature_frame(fight_df)
    X_pred = feature_frame.reindex(columns=bundle["feature_columns"], fill_value=np.nan)

    prob_red = float(bundle["model"].predict_proba(X_pred)[0, 1])
    prob_blue = 1 - prob_red
    red_name = fight_df.iloc[0].get("RedFighter", "Red")
    blue_name = fight_df.iloc[0].get("BlueFighter", "Blue")

    predicted_corner = "Red" if prob_red >= prob_blue else "Blue"
    predicted_winner = red_name if predicted_corner == "Red" else blue_name
    confidence = prob_red if predicted_corner == "Red" else prob_blue

    red_decimal = float(american_to_decimal([fight_df.iloc[0]["RedOdds"]])[0])
    blue_decimal = float(american_to_decimal([fight_df.iloc[0]["BlueOdds"]])[0])
    ev_red = prob_red * red_decimal - 1
    ev_blue = prob_blue * blue_decimal - 1

    if ev_red >= ev_blue:
        bet_corner = "Red"
        bet_name = red_name
        best_ev = ev_red
    else:
        bet_corner = "Blue"
        bet_name = blue_name
        best_ev = ev_blue

    recommended_bet = bet_name if best_ev > bundle["threshold"] else "Pass"

    return pd.Series(
        {
            "weight_class_model_used": bundle["weight_class"] or "GLOBAL_FALLBACK",
            "estimator": bundle["model_name"],
            "model_params": bundle["params_label"],
            "predicted_winner": predicted_winner,
            "confidence": confidence,
            "prob_red": prob_red,
            "prob_blue": prob_blue,
            "expected_value_red": ev_red,
            "expected_value_blue": ev_blue,
            "bet_threshold": bundle["threshold"],
            "recommended_bet": recommended_bet,
            "recommended_bet_corner": bet_corner if recommended_bet != "Pass" else "Pass",
        }
    )


REQUIRED_PRE_FIGHT_COLUMNS = [c for c in BASE_FEATURE_COLS if c != "Date"]
display(pd.Series(REQUIRED_PRE_FIGHT_COLUMNS, name="required_columns"))


,segment,model,params_label,tuning_year,deployment_threshold
0,Bantamweight,gb,"learning_rate=0.04, max_depth=3, n_estimators=250",2024,-0.0200
1,Catch Weight,gb,"learning_rate=0.04, max_depth=3, n_estimators=250",2024,0.1000
2,Featherweight,logreg,C=0.1,2024,0.1950
3,Flyweight,hist_gb,"learning_rate=0.02, max_depth=6, max_iter=500",2024,-0.0200
4,Heavyweight,logreg,C=0.5,2024,0.1550
5,Light Heavyweight,rf,"max_depth=None, min_samples_leaf=2, n_estimato...",2024,0.1900
6,Lightweight,extra_trees,"max_depth=None, min_samples_leaf=3, n_estimato...",2024,0.0200
7,Middleweight,xgb,"learning_rate=0.03, max_depth=3, n_estimators=350",2024,-0.0200
8,Welterweight,rf,"max_depth=None, min_samples_leaf=3, n_estimato...",2024,0.1400
9,Women's Bantamweight,ada,"learning_rate=0.05, n_estimators=200",2024,0.0950


0                   RedOdds
1                  BlueOdds
2                  OddsDiff
3                   AgeDiff
4                 ReachDiff
5                HeightDiff
6                  WinsDiff
7                LossesDiff
8                RoundsDiff
9             WinStreakDiff
10           LoseStreakDiff
11                 RankDiff
12                TitleBout
13              WeightClass
14                   Gender
15           NumberOfRounds
16    BlueCurrentLoseStreak
17     BlueCurrentWinStreak
18     BlueLongestWinStreak
19               BlueLosses
20    BlueTotalRoundsFought
21      BlueTotalTitleBouts
22             BlueWinsByKO
23     BlueWinsBySubmission
24                 BlueWins
25               BlueStance
26            BlueHeightCms
27             BlueReachCms
28                  BlueAge
29             BMatchWCRank
30     RedCurrentLoseStreak
31      RedCurrentWinStreak
32      RedLongestWinStreak
33                RedLosses
34     RedTotalRoundsFought
35       RedTotalTit

## Using The Final Models

1. Build a one-row `DataFrame` that matches the pre-fight schema from `df.csv`.
2. Call `predict_fight(your_row)`.
3. Read `predicted_winner` for the straight winner pick and `confidence` for the `0-1` confidence score.
4. If you care about betting value instead of just winner prediction, use `recommended_bet` and the expected value columns.

Example:

```python
upcoming_fight = pd.DataFrame([
    {
        "RedFighter": "Example Red",
        "BlueFighter": "Example Blue",
        "RedOdds": -145,
        "BlueOdds": 122,
        "OddsDiff": -267,
        "AgeDiff": 2,
        "ReachDiff": 5.08,
        "HeightDiff": 2.54,
        "WinsDiff": 4,
        "LossesDiff": -1,
        "RoundsDiff": 18,
        "WinStreakDiff": 2,
        "LoseStreakDiff": 0,
        "RankDiff": -3,
        "WeightClass": "Featherweight",
        "Gender": "MALE",
        "TitleBout": False,
        "NumberOfRounds": 3,
        "BlueCurrentLoseStreak": 0,
        "BlueCurrentWinStreak": 2,
        "BlueLongestWinStreak": 4,
        "BlueLosses": 3,
        "BlueTotalRoundsFought": 26,
        "BlueTotalTitleBouts": 0,
        "BlueWinsByKO": 3,
        "BlueWinsBySubmission": 2,
        "BlueWins": 8,
        "BlueStance": "Orthodox",
        "BlueHeightCms": 177.8,
        "BlueReachCms": 177.8,
        "BlueAge": 29,
        "BMatchWCRank": 10,
        "RedCurrentLoseStreak": 0,
        "RedCurrentWinStreak": 4,
        "RedLongestWinStreak": 6,
        "RedLosses": 2,
        "RedTotalRoundsFought": 44,
        "RedTotalTitleBouts": 1,
        "RedWinsByKO": 5,
        "RedWinsBySubmission": 2,
        "RedWins": 12,
        "RedStance": "Orthodox",
        "RedHeightCms": 180.34,
        "RedReachCms": 182.88,
        "RedAge": 31,
        "RMatchWCRank": 7,
    }
])

predict_fight(upcoming_fight)
```


In [7]:
upcoming_fight = pd.DataFrame([
    {
        "RedFighter": "Jiri Prochazka",
        "BlueFighter": "Carlos Ulberg",
        "RedOdds": -125,
        "BlueOdds": 105,
        "OddsDiff": -230,
        "AgeDiff": -2,
        "ReachDiff": 7.78,
        "HeightDiff": -2.54,
        "WinsDiff": -4,
        "LossesDiff": 1,
        "RoundsDiff": 1,
        "WinStreakDiff": -7,
        "LoseStreakDiff": 0,
        "RankDiff": 1,
        "WeightClass": "Light Heavyweight",
        "Gender": "MALE",
        "TitleBout": True,
        "NumberOfRounds": 5,
        "BlueCurrentLoseStreak": 0,
        "BlueCurrentWinStreak": 9,
        "BlueLongestWinStreak": 9,
        "BlueLosses": 1,
        "BlueTotalRoundsFought": 20,
        "BlueTotalTitleBouts": 0,
        "BlueWinsByKO": 6,
        "BlueWinsBySubmission": 1,
        "BlueWins": 10,
        "BlueStance": "Orthodox",
        "BlueHeightCms": 190.5,
        "BlueReachCms": 195.58,
        "BlueAge": 35,
        "BMatchWCRank": 3,
        "RedCurrentLoseStreak": 0,
        "RedCurrentWinStreak": 2,
        "RedLongestWinStreak": 3,
        "RedLosses": 2,
        "RedTotalRoundsFought": 21,
        "RedTotalTitleBouts": 3,
        "RedWinsByKO": 5,
        "RedWinsBySubmission": 1,
        "RedWins": 6,
        "RedStance": "Orthodox",
        "RedHeightCms": 187.96,
        "RedReachCms": 203.2,
        "RedAge": 33,
        "RMatchWCRank": 2,
    }
])

predict_fight(upcoming_fight)

weight_class_model_used                                    Light Heavyweight
estimator                                                                 rf
model_params               max_depth=None, min_samples_leaf=2, n_estimato...
predicted_winner                                               Carlos Ulberg
confidence                                                            0.5208
prob_red                                                              0.4792
prob_blue                                                             0.5208
expected_value_red                                                   -0.1375
expected_value_blue                                                   0.0677
bet_threshold                                                         0.1900
recommended_bet                                                         Pass
recommended_bet_corner                                                  Pass
dtype: object